In [1]:
from jpx_ranker.kaggle_setup import setup_kaggle_credentials, download_competition_data, download_dataset

# Setup credentials from .env file
setup_kaggle_credentials()
print("✓ Kaggle authentication configured")

INFO: Pandarallel will run on 11 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.
✓ Kaggle credentials loaded for user: dsxavier
✓ Kaggle authentication configured


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

# Download competition data
jpx_tokyo_stock_exchange_prediction_path = download_competition_data('jpx-tokyo-stock-exchange-prediction')

# Download additional datasets
dsxavier_jpx_pre_path = download_dataset('dsxavier/jpx-pre')
dsxavier_jpx_deps_v1_path = download_dataset('dsxavier/jpx-deps-v1')

print('Data source import complete.')



# Dependencies

## Install Dependencies

In [ ]:
!nvidia-smi

In [ ]:
!pip install -qq optuna catboost skfolio loguru

In [ ]:
dps_path = '/kaggle/input/jpx-deps-v1'
# deps
# - pandarallel
# - cudf-cu12 (Pandas Accelerator)

!pip install -qq --no-index --find-links {dps_path}/jpx-deps -r {dps_path}/requirements.txt
# get_ipython().kernel.do_shutdown(restart=True) # Kaggle disable ONLY

In [ ]:
%config Completer.use_jedi = False

## Import Dependencies

In [2]:
# %load_ext cudf.pandas # Kaggle disable ONLY
import os
import warnings
import multiprocessing as mp
from pathlib import Path
from typing import Optional, Union, Tuple, List, Any
from collections import defaultdict

import pandas as pd
import numpy as np


from pandarallel import pandarallel as ppl
ppl.initialize(progress_bar=False, nb_workers=mp.cpu_count() - 1)

from tqdm.notebook import tqdm
from IPython.display import clear_output, display

from statsmodels.tsa.stattools import adfuller
from sklearn.base import BaseEstimator
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import make_scorer, mean_absolute_error, mean_absolute_percentage_error
from skfolio.model_selection import CombinatorialPurgedCV

import optuna
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

import matplotlib.pylab as plt
import seaborn as sns

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)

os.environ['PYTHONHASHSEED'] = str(42)
np.random.seed(42)

INFO: Pandarallel will run on 11 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


# Data Preprocessing & Analysis

## Data Ingestion

In [ ]:
stk_prc = pd.read_csv('../kaggle/input/jpx-tokyo-stock-exchange-prediction/train_files/stock_prices.csv')
stk_prc

In [ ]:
stk_prc = stk_prc.drop(columns=['RowId', 'AdjustmentFactor', 'ExpectedDividend', 'SupervisionFlag'])
stk_prc

In [ ]:
df = stk_prc.copy(deep=True)

## Data Preprocessing & Analysis

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df = df[['Date', 'SecuritiesCode', 'Close', 'Volume', 'Target']].dropna().reset_index(drop=True)
df

In [ ]:
tmp_df = df[df['SecuritiesCode'] == 4202]
tmp_df.Close.plot()

We must select events by comparing the shifted mean average to the target value using the cumulative buy/sell directions, subject to a threshold. To achieve this, we will employ an event-based sampling technique known as the CUMSUM Filter (Lopez de Parto, 2018). However, before implementing this method, it is essential to calculate both the volatility of each security and their average volatility.

In [ ]:
def get_daily_vol(close: pd.Series, span=100) -> pd.Series:
    """
    Compute the daily volatility of a time series.

    This function calculates the daily volatility of a time series using
    an exponentially weighted moving standard deviation (EWMSD) approach.

    Parameters
    ----------
    close : pd.Series
        Series containing the closing prices of the asset.
    span : int, optional
        Span parameter for the EWMSD calculation.
        Default is 100.

    Returns
    -------
    pd.Series
        Series containing the daily volatility.
    """
    # daily vol, reindexed to close
    df = close.index.searchsorted(close.index - pd.Timedelta(days=1))
    df = df[df > 0]
    df = pd.Series(close.index[df - 1], index=close.index[close.shape[0] - df.shape[0]:], name='sd').to_frame()
    df = (close.loc[df.index] / close.loc[df.sd].values) - 1 # daily return
    df = df.ewm(span=span).std()
    return df

In [ ]:
def get_events(df: pd.DataFrame, threshold: Union[int, float]=0.5) -> pd.DatetimeIndex:
    """
    Identify events in a time series based on a CUMSUM filter.

    This function detects events in a time series by applying a CUMSUM filter
    to a specified target column.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing the time series data.
    threshold : Union[int, float], optional
        Threshold for detecting events.
        Default is 0.5.

    Returns
    -------
    pd.DatetimeIndex
        DatetimeIndex containing the timestamps of detected events.
    """
    events_t, s_pos, s_neg = [], 0, 0
    diff = df.Target
    for idx in diff.index:
        s_pos, s_neg = max(0, s_pos + diff.loc[idx]), min(0, s_neg + diff.loc[idx])

        if s_neg < -threshold:
            s_neg = 0; events_t.append(idx)
        elif s_pos > threshold:
            s_pos = 0; events_t.append(idx)

    return pd.DatetimeIndex(events_t)

Next, we aim to implement the triple-barrier method to categorize observations using predefined profit-taking and stop-loss thresholds. This approach is valuable for establishing profit and loss boundaries, as well as for determining return weights. Additionally, by incorporating this method, the model can identify securities that are expected to yield higher returns or experience reduced losses within a specified time frame.

In [ ]:
def apply_put_sell_on_threshold(events, close, put_sell) -> pd.DataFrame:
    r"""
    Apply put option selling strategy based on given threshold values.

    This function applies a put option selling strategy to events based on specified
    threshold values for profit taking (pt) and stop loss (sl).

    Parameters
    ----------
    events : pandas.DataFrame
        DataFrame containing event information, including ``tv1`` (end of event) and
        ``trgt`` (target value).
    close : pandas.Series
        Series containing closing prices.
        This series is used to calculate path prices and returns.
    put_sell : tuple
        Tuple containing threshold values for profit taking (pt) and stop loss (sl).
        The first element represents the profit taking threshold, and
        the second element represents the stop loss threshold.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing the earliest profit taking ('pt') and
        stop loss ('sl') timestamps for each event.

    Notes
    -----
    The function calculates path returns based on the closing prices from the event
    start to the end.
    Profit taking (pt) and stop loss (sl) thresholds are applied to the path returns
    to determine the respective timestamps.
    """
    # apply stop loss/profit taking, if it takes place before tv1 (end of event)
    out = events[['tv1']].copy(deep=True)
    if put_sell[0] > 0:
        pt = put_sell[0] * events['trgt']
    else:
        pt = pd.Series(index=[events.name], dtype=np.float32) # `np.nan` values

    if put_sell[1] > 0:
        sl = -put_sell[1] * events['trgt']
    else:
        sl = pd.Series(index=[events.name], dtype=np.float32) # `np.nan` values

    loc_ = events.name
    tv1 = events['tv1']

    df = close[loc_: tv1] # path prices
    df = ((df / close[loc_]) - 1) * events['side'] # path returns

    if isinstance(pt, float):
        src_ = df[df > pt].tolist()
        out['pt'] = np.min(src_) if len(src_) > 0 else np.nan
    else:
        out['pt'] = df[df > pt[loc_]].index.min() # earliest profit taking

    if isinstance(sl, float):
        src_ = df[df < sl]
        out['sl'] = np.min(src_) if len(src_) > 0 else np.nan
    else:
        out['sl'] = df[df < sl[loc_]].index.min() # earliest stop loss.
    return out

In [ ]:
def get_vert_barrier(close: pd.Series,
                     t_events: pd.DatetimeIndex,
                     num_days: int) -> pd.Series:
    """
    Add a vertical barrier to events.

    Parameters
    ----------
    close : pd.Series
        Close prices of the security for given datetimes.
    t_events : pd.DatetimeIndex
        Timestamps representing event occurrences.
    num_days : int
        Number of days to define the vertical barrier.

    Returns
    -------
    pd.Series
        Timestamps indicating when the first vertical barrier is touched.

    Notes
    -----
    This function finds the timestamp of the next price bar at or immediately
    after a specified number of days from each event.
    The vertical barrier is defined as the time when the price reaches the specified
    number of days from the event timestamp.
    """
    tv1 = close.index.searchsorted(t_events + pd.Timedelta(days=num_days))
    tv1 = tv1[tv1 < close.shape[0]]
    tv1 = pd.Series(close.index[tv1], index=t_events[:tv1.shape[0]], name='tv1')  # NaNs at end
    return tv1

In [ ]:
def get_tbl_events(close: pd.Series,
                   t_events: pd.DatetimeIndex,
                   put_sell: Tuple[int, int],
                   trgt: pd.Series,
                   min_ret: float,
                   tv1: Union[bool, pd.Series] = False,
                   side: Optional[pd.Series] = None) -> pd.Series:
    r"""
    Generate a table of events based on specified parameters.

    Parameters
    ----------
    close : pd.Series
        Close prices of the security for given datetimes.
    t_events : pd.DatetimeIndex
        Timestamps representing event occurrences.
    put_sell : Tuple[int, int]
        Tuple containing threshold values for profit taking (pt) and stop loss (sl).
        The first element represents the profit taking threshold,
        and the second element represents the stop loss threshold.
    trgt : pd.Series
        Series containing target values.
        These values are used for filtering events based on minimum returns/vol.
    min_ret : float
        Minimum return/vol threshold for filtering events.
        Events with target values below this threshold are discarded.
    tv1 : Union[bool, pd.Series], optional
        Series containing the timestamp of the next price bar at or immediately
        after a number of days. If False, no vertical barrier is applied. Defaults to ``False``.
    side : Optional[pd.Series], optional
        Series containing side information for each event.
        This parameter is used to specify the side of each event (e.g., 1 for long, -1 for short).
        Defaults to ``None``.

    Returns
    -------
    pd.Series
        Table of events containing the timestamp of the vertical barrier (``tv1``),
        target values (``trgt``), profit taking timestamps (``pt``),
        and stop loss timestamps (``sl``).

    Notes
    -----
    Events are filtered based on minimum return thresholds and optionally by side information.
    The vertical barrier, profit taking, and stop loss are applied to each event according
    to the specified thresholds.
    """
    # 1) get targets
    com_idx = t_events.intersection(trgt.index)
    trgt = trgt.loc[com_idx] # sampling data based on CUM Filtering
    trgt = trgt.loc[trgt > min_ret] # filter by the min returns

    # 2) get tv1 (max holding period)
    if tv1 is False:
        tv1 = pd.Series(pd.NaT)

    # 3) form events object, apply stop loss on tv1
    if side is None:
        side_, pt_sl_ = pd.Series(1., index=trgt.index), [put_sell[0], put_sell[0]]
        events = pd.concat({'tv1': tv1, 'trgt': trgt,
                            'side': side_},
                        axis=1).dropna(subset=['trgt'])
    else:
        com_idx = side.index.intersection(trgt.index)
        side_, pt_sl_ = side.loc[com_idx], put_sell[:2]
        events = pd.concat({'tv1': tv1, 'trgt': trgt,
                            'side': side_},
                            axis=1).dropna(subset=['trgt', 'side'])

    events['tv1'] = events['tv1'].fillna(close.index[-1])

    df = events.parallel_apply(apply_put_sell_on_threshold, args=(close, pt_sl_), axis=1)
    events['pt'] = df['pt']
    events['sl'] = df['sl']
    events['tv1'] = df.dropna(how='all')['tv1']
    if side is None:
        events = events.drop('side', axis=1)

    return events

In [ ]:
def compute_tbl_events(df: pd.DataFrame,
                       put_sell: Tuple[int, int],
                       num_days: int = 1,
) -> pd.DataFrame:
    r"""
    Compute Triple Barriers events for each security in a DataFrame.

    This function computes Triple Barrier Labeling (TBL) events for each security
    in the provided DataFrame. It iterates over each unique ``SecuritiesCode`` and
    calculates TBL events based on the given parameters.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing the time series data for multiple securities.
    put_sell : Tuple[int, int]
        Tuple containing the multipliers for the lower and upper barriers, respectively.
    num_days : int, optional
        Number of days to wait before closing the position, by default ``1``.

    Returns
    -------
    Tuple[pd.DataFrame, pd.Series]
            - pd.DataFrame: DataFrame containing the TBL events.
    """

    df_ = pd.DataFrame()
    lst_secs = df.SecuritiesCode.unique().tolist()
    prg_bar = tqdm(enumerate(lst_secs), total=len(lst_secs),
                   desc=f'Compute Triple barriers for {num_days} position')
    for idx, sec in prg_bar:
        sec_df = df[df['SecuritiesCode'] == sec].reset_index(drop=True)
        sec_df['Date'] = pd.to_datetime(sec_df['Date'])
        sec_df = sec_df.set_index('Date')

        vol = get_daily_vol(sec_df.Close)
        avg_vol = vol.mean()

        t_events = get_events(sec_df, avg_vol)
        t1 = get_vert_barrier(sec_df.Close, t_events, num_days)
        print(f"sec: {sec}, barrier len: {t1.shape[0]}")
        if t1.shape[0] > 50:
            events = get_tbl_events(sec_df.Close, t_events, put_sell,
                                    vol, avg_vol, t1)
            sec_df = sec_df.loc[events.index]
            sec_df = sec_df.reset_index().dropna()
            df_ = pd.concat([df_, sec_df])

        if idx % 5 == 0:
            clear_output(wait=True)
            display(prg_bar.container)
    df_ = df_.sort_values(by='index').reset_index(drop=True)
    df_.columns = df_.columns[1:].insert(0, 'Date')
    return df_

Prior to computing the triple barrier, we had already stored the outcomes in a private dataset because of issues encountered during parallel computation with Kaggle. Consequently, to save time, we executed the code on Colab, where it completed processing within 1 hour and 7 minutes, utilizing only one processor. Consequently, I determined that it would be more efficient to store the results in a dataset rather than recompute them during debugging and notebook commits.

In [ ]:
put_sell = (1, 1)
one_day = 1
two_days = 2
df_1, df_2 = None, None
data_path = Path('../kaggle/input/jpx-pre/versions/2/')
os.makedirs(data_path, exist_ok=True)
file_name = 'tbm_data.csv'

if os.path.exists(data_path / file_name):
    df = pd.read_csv(data_path / file_name)
else:
    df_1 = compute_tbl_events(df, put_sell, one_day)
    df_2 = compute_tbl_events(df, put_sell, two_days)
    df = pd.concat([df_1, df_2])
    df = df[~df.duplicated(keep='first')]
    df = df.dropna().reset_index(drop=True)
    df.to_csv(data_path / file_name, index=False)
df

Review the securities and eliminate those with fewer than 50 samples or 1.9E-4 percent. This strategy effectively removes the rarest samples, thereby preventing them from unduly influencing the model weights.

In [ ]:
df.SecuritiesCode.value_counts()

In [ ]:
df.SecuritiesCode.value_counts(normalize=True)

In [ ]:
def drop_securities(events: pd.DataFrame, min_pct: float = 1.9e-4, max_drop_sec: int = 400):
    r"""
    Review securities and eliminate those with fewer than a specified percentage of samples.

    This function reviews the securities in the provided DataFrame and
    eliminates those with fewer samples than the specified percentage.
    It iterates until all securities meet the minimum sample percentage criterion.

    Parameters
    ----------
    events : pd.DataFrame
        DataFrame containing the securities data.
    min_pct : float, optional
        Minimum percentage of samples allowed for each security, by default ``1.9e-4``.
    max_drop_sec : int, optional
        Maximum number of securities to drop, by default ``400``.
        We can't go less to fit the ranking system of the competition.

    Returns
    -------
    pd.DataFrame
        DataFrame containing the remaining securities after elimination
        based on sample percentage.
    """
    # apply weights, drop labels with insufficient examples
    events_ = events
    while True:
        df = events_['SecuritiesCode'].value_counts(normalize=True)
        if df.min() > min_pct or df.shape[0] < max_drop_sec : break
        print(f'Dropped Label ({df.idxmin()}, {df.min()})')
        events_ = events_[events_['SecuritiesCode'] != df.idxmin()]

    return events_

In [ ]:
df = drop_securities(df)

In [ ]:
df.SecuritiesCode.value_counts(normalize=True)

Now that we have calculated the triple barrier events for each security, we need to recalculate the target returns to account for the alterations in the events we have identified.

In [ ]:
def cal_target(df):
    """
    Calculate target returns based on given periods (1-day return, 2-day returns).

    This function calculates target returns for each security in the provided DataFrame.
    It computes the percentage change in the closing price between the current day and the next day,
    and between the current day and the day after next, representing 1-day and 2-day returns respectively.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing the time series data for multiple securities.

    Returns
    -------
    pd.DataFrame
        DataFrame containing the calculated target returns for each security.
    """
    df_ = pd.DataFrame()
    lst_secs = df.SecuritiesCode.unique().tolist()
    prg_bar = tqdm(enumerate(lst_secs), total=len(lst_secs),
                   desc=f'Compute Target Returns')
    for _, sec in prg_bar:
        sec_df = df[df['SecuritiesCode'] == sec].reset_index(drop=True)
        sec_df['Date'] = pd.to_datetime(sec_df['Date'])
        sec_df = sec_df.set_index('Date')

        # ref: https://www.kaggle.com/code/chumajin/easy-to-understand-the-competition?scriptVersionId=94143164&cellId=17
        sec_df["Close_shift1"] = sec_df["Close"].shift(-1)
        sec_df["Close_shift2"] = sec_df["Close"].shift(-2)
        sec_df["Target"] = (sec_df["Close_shift2"] - sec_df["Close_shift1"]) / sec_df["Close_shift1"]
        sec_df = sec_df[['SecuritiesCode', 'Close', 'Volume', 'Target']]

        sec_df = sec_df.dropna().reset_index()
        df_ = pd.concat([df_, sec_df])
    df_ = df_.sort_values(by='Date').reset_index(drop=True)
    return df_

In [ ]:
df = cal_target(df)
df

Let's have the volume weighted average price (VWAP) of each equity

In [ ]:
from _src.utils import compute_vwap

In [ ]:
df = compute_vwap(df)
df

In [ ]:
df[df.SecuritiesCode == 4202]

In [ ]:
tmp_df = df[df['SecuritiesCode'] == 4202]
plt.plot(tmp_df['Close'])
plt.show()

In [ ]:
plt.plot(tmp_df['VWAP'])
plt.show()

After comparing the charts for the selected security before and after applying our preprocessing steps, we observe a significant change in distribution. This improved distribution, achieved through our preprocessing, effectively targets long and short positions for entering and exiting the market for each security. Consequently, training a model to rank these positions becomes critical for guiding investor decisions and determining which securities to invest in or take a short position on, thereby refining the model's decision boundary and enhancing precision.

Now, let's conduct a stationarity test on both the `Target` and `rate_vwap` variables to determine which one to utilize in our subsequent processes.


In [ ]:
stats = adfuller(tmp_df.Target)
t_test, p_value = stats[:2]
t_test, p_value

In [ ]:
stats = adfuller(tmp_df.VWAP)
t_test, p_value = stats[:2]
t_test, p_value

## Feature Engineering

In [ ]:
from _src.utils import compute_feature_eng

In [ ]:
fast_period = 5
slow_period = 10

df = compute_feature_eng(df, fast_period, slow_period)
df

In [ ]:
df = df[['Date', 'SecuritiesCode', 'Target', 'vwap_rate', 'side', 'vol5', 'vol10', 'vol15', 'vol21', 'autocorr-10', 'autocorr-15', 'autocorr-21']]
df['side'] = df['side'].astype(int)

# Compute correlation across all data (not grouped by SecuritiesCode and Date)
# since groupby().corr() on those columns leaves no variation within groups
df_vis = df[['Target', 'vwap_rate', 'side', 'vol5', 'vol10', 'vol15', 'vol21', 'autocorr-10', 'autocorr-15', 'autocorr-21']].corr()

plt.figure(figsize=(15, 10))
sns.heatmap(df_vis, annot=True, fmt='.2f')
plt.show()

In [ ]:
correlation_df = df.groupby('SecuritiesCode').corr().droplevel(0)
correlation_df

In [ ]:
plt.figure(figsize=(15, 10))
plt.imshow(correlation_df.values, cmap='viridis', aspect='auto')
plt.colorbar(label='Correlation')
plt.xticks(ticks=[])
plt.yticks(ticks=[])
plt.title('Correlation Heatmap')
plt.xlabel('Features')
plt.ylabel('Features')
plt.show()

In [ ]:
df = df[['Date', 'SecuritiesCode', 'Target', 'vwap_rate', 'side', 'vol5', 'vol10', 'vol15', 'vol21', 'autocorr-10', 'autocorr-15', 'autocorr-21']]
df['Rank'] = df.groupby("Date")["Target"].rank(ascending=False,method="first") - 1
df['Rank'] = df['Rank'].astype('int')
df

In [ ]:
df.info()

In [ ]:
df = df.reset_index(drop=True)
os.makedirs('../kaggle/working/', exist_ok=True)
df.to_feather('../kaggle/working/preped_data.feather')

# Data Prepration & Modeling

In [2]:
from jpx_ranker._src.utils import compute_vwap
from jpx_ranker._src.utils import compute_feature_eng

INFO: Pandarallel will run on 11 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [3]:
df = pd.read_feather('kaggle/working/preped_data.feather')
df

,Date,SecuritiesCode,Target,vwap_rate,side,vol5,vol10,vol15,vol21,autocorr-10,autocorr-15,autocorr-21,Rank
0,2017-01-12,3902,-0.045499,-0.065895,1,0.086390,0.086390,0.086390,0.086390,-1.000000,-1.000000,-1.000000,0
1,2017-01-13,2393,-0.063366,-0.002023,1,0.041620,0.041620,0.041620,0.041620,-1.000000,-1.000000,-1.000000,4
2,2017-01-13,3778,-0.005168,0.002756,1,0.083890,0.083890,0.083890,0.083890,1.000000,1.000000,1.000000,2
3,2017-01-13,8291,0.024024,0.000079,1,0.020683,0.020683,0.020683,0.020683,-1.000000,-1.000000,-1.000000,0
4,2017-01-13,6904,-0.047386,-0.004580,1,0.069681,0.069681,0.069681,0.069681,1.000000,1.000000,1.000000,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
249154,2021-11-26,7169,0.049421,-0.003565,1,0.058295,0.107104,0.092151,0.086414,0.348258,0.385422,0.433330,0
249155,2021-11-29,2158,-0.194444,0.000322,-1,0.157201,0.195384,0.162123,0.141544,0.121414,0.135910,0.122663,3
249156,2021-11-29,2931,-0.033962,-0.001259,-1,0.021157,0.072776,0.061889,0.054842,-0.262161,-0.195706,-0.173842,2
249157,2021-11-29,6912,0.050420,0.000889,-1,0.165028,0.117723,0.100211,0.088920,-0.240556,-0.188016,-0.065297,0


In [4]:
df_test = pd.read_csv('kaggle/input/jpx-tokyo-stock-exchange-prediction/supplemental_files/stock_prices.csv')
df_test

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target
0,20211206_1301,2021-12-06,1301,2982.0,2982.0,2965.0,2971.0,8900,1.0,NaN,False,-0.003263
1,20211206_1332,2021-12-06,1332,592.0,599.0,588.0,589.0,1360800,1.0,NaN,False,-0.008993
2,20211206_1333,2021-12-06,1333,2368.0,2388.0,2360.0,2377.0,125900,1.0,NaN,False,-0.009963
3,20211206_1375,2021-12-06,1375,1230.0,1239.0,1224.0,1224.0,81100,1.0,NaN,False,-0.015032
4,20211206_1376,2021-12-06,1376,1339.0,1372.0,1339.0,1351.0,6200,1.0,NaN,False,0.002867
...,...,...,...,...,...,...,...,...,...,...,...,...
269876,20220624_9990,2022-06-24,9990,576.0,576.0,563.0,564.0,24200,1.0,NaN,False,0.027073
269877,20220624_9991,2022-06-24,9991,810.0,815.0,804.0,815.0,8700,1.0,NaN,False,0.001220
269878,20220624_9993,2022-06-24,9993,1548.0,1548.0,1497.0,1497.0,12600,1.0,NaN,False,0.001329
269879,20220624_9994,2022-06-24,9994,2507.0,2527.0,2498.0,2527.0,7300,1.0,NaN,False,0.003185


In [5]:
fast_period = 5
slow_period = 10

# Check if the test preprocessing data is already exists previously:
if os.path.exists('kaggle/working/test_preped_data.feather'):
    df_fet_test = pd.read_feather('kaggle/working/test_preped_data.feather')
else:
    df_fet_test = df_fet_test[['Date', 'SecuritiesCode', 'Close', 'Volume', 'Target']].dropna().reset_index(drop=True)
    df_fet_test = compute_vwap(df_fet_test)
    df_fet_test = compute_feature_eng(df_fet_test, fast_period, slow_period)
    df_fet_test = df_fet_test[['Date', 'SecuritiesCode', 'Target', 'vwap_rate', 'side', 'vol5', 'vol10', 'vol15', 'vol21', 'autocorr-10', 'autocorr-15', 'autocorr-21']]
    df_fet_test['Rank'] = df_fet_test.groupby("Date")["Target"].rank(ascending=False,method="first") - 1
    df_fet_test['Rank'] = df_fet_test['Rank'].astype('int')
    df_fet_test.to_feather('../kaggle/working/test_preped_data.feather')

df_fet_test

,Date,SecuritiesCode,Target,vwap_rate,side,vol5,vol10,vol15,vol21,autocorr-10,autocorr-15,autocorr-21,Rank
0,2021-12-08,1301,0.006483,0.003943,1.0,0.006798,0.006798,0.006798,0.006798,-1.000000,-1.000000,-1.000000,229
1,2021-12-08,9039,-0.001149,-0.001976,1.0,0.011513,0.011513,0.011513,0.011513,-1.000000,-1.000000,-1.000000,585
2,2021-12-08,7581,-0.009804,-0.000126,1.0,0.010906,0.010906,0.010906,0.010906,-1.000000,-1.000000,-1.000000,1129
3,2021-12-08,2175,-0.016985,0.007379,1.0,0.033579,0.033579,0.033579,0.033579,-1.000000,-1.000000,-1.000000,1456
4,2021-12-08,6742,0.030238,0.019414,1.0,0.015312,0.015312,0.015312,0.015312,-1.000000,-1.000000,-1.000000,24
...,...,...,...,...,...,...,...,...,...,...,...,...,...
261020,2022-06-22,4471,0.012917,-0.000462,1.0,0.016992,0.013699,0.011785,0.012006,-0.246049,-0.150591,-0.117192,833
261021,2022-06-22,6823,0.007475,0.000067,1.0,0.014206,0.024808,0.022947,0.020255,-0.576017,-0.464550,-0.360416,1077
261022,2022-06-22,3854,0.014013,0.000962,1.0,0.024624,0.026386,0.043903,0.038565,0.206552,-0.292716,-0.315033,769
261023,2022-06-22,7516,0.012517,0.000080,1.0,0.010558,0.011919,0.013504,0.012557,0.116348,-0.079125,-0.091047,848


After preprocessing our dataset and incorporating additional features, the next step is to prepare our dataset for training with a ranking model. The ranking model enables us to train models based on the extracted features and obtain the rank of each security's features. 

To avoid overfitting and data leakage in financial time series, we utilize an advanced cross-validation technique called **CombinatorialPurgedCV** from [skfolio](https://skfolio.org/generated/skfolio.model_selection.CombinatorialPurgedCV.html), which is specifically designed for financial machine learning. This approach offers significant advantages over standard `TimeSeriesSplit`:

- **Purging**: Removes training observations whose labels overlap in time with test labels, preventing leakage from multi-day return calculations
- **Embargoing**: Excludes observations immediately following test periods to handle serial correlation in financial features (ARMA processes, momentum effects)
- **Combinatorial Paths**: Creates multiple train/test combinations for more robust validation (e.g., C(10,2) = 45 splits instead of just 5)

Our implementation trains three state-of-the-art ranking models (`LGBMRanker`, `XGBRanker`, and `CatBoostRanker`) using **Optuna** for hyperparameter optimization. The best performing model is automatically selected based on validation NDCG@100 scores.

Additionally, we've developed a modular, professional pipeline architecture with:
- **DataProcessor**: Unified data preparation for all three frameworks
- **ModelSelector**: Automated model training, comparison, and selection
- **TrainingPipeline**: Complete training orchestration with evaluation
- **InferencePipeline**: Kaggle-compatible submission generation

This approach follows best practices from "Advances in Financial Machine Learning" by Marcos López de Prado, ensuring robust and production-ready algorithmic trading models.

In [6]:
train = df.copy(deep=True)
test = df_fet_test.copy(deep=True)

In [7]:
from jpx_ranker._src.utils import calc_spread_return_sharpe
from jpx_ranker._src.utils import calc_spread_return_sharpe_scorer

## Data Preparation

In [8]:
from jpx_ranker._src.data_prep import DataProcessor

## Model Training Pipeline

In [9]:
from jpx_ranker._src.train import TrainingPipeline

In [10]:
TRAIN_CONFIG = {
    'test_size': 0.2,          # Validation split ratio (20% for validation)
    'n_trials': 5, #200,           # Optuna trials per model (increase for better optimization)
    'n_folds': 5, #10,             # Number of folds for CombinatorialPurgedCV
    'n_test_folds': 2,         # Number of test folds per split
    'purged_size': 2,          # Days to purge around test sets (prevents label leakage)
    'embargo_size': 21,         # Days to embargo after test sets (handles serial correlation)
    'seed': 42,                # Random seed for reproducibility
    'save_model_path': 'kaggle/working/best_ranker_model.pkl',
    'device': 'cpu',           # Change to 'cuda' when submitting to Kaggle with GPU
    'min_trials': 5            # Minimum number of trials before checking for early stopping
}

# Feature engineering parameters (if needed)
UPDATED_FEATURE_CONFIG = {
    'fast_period': 5,
    'slow_period': 10
}

print("✓ Updated Configuration Loaded")
print(f"  Training config: {TRAIN_CONFIG}")
print(f"  Feature config: {UPDATED_FEATURE_CONFIG}")
print(f"\n  Note: Set n_trials=200 for production, n_trials=3 for quick testing")

✓ Updated Configuration Loaded
  Training config: {'test_size': 0.2, 'n_trials': 5, 'n_folds': 5, 'n_test_folds': 2, 'purged_size': 2, 'embargo_size': 21, 'seed': 42, 'save_model_path': 'kaggle/working/best_ranker_model.pkl', 'device': 'cpu', 'min_trials': 5}
  Feature config: {'fast_period': 5, 'slow_period': 10}

  Note: Set n_trials=200 for production, n_trials=3 for quick testing


In [11]:
# Updated Training Pipeline with Fixed Model Names and Result Keys
# This cell uses the corrected configuration and properly accesses result dictionary keys

print("="*80)
print("STARTING MODEL TRAINING WITH UPDATED PIPELINE")
print("="*80)
print(f"Training data shape: {df.shape}")
print(f"Features: {df.columns.tolist()}\n")

# Initialize the training pipeline with updated configuration
pipeline = TrainingPipeline(
    test_size=TRAIN_CONFIG['test_size'],
    n_trials=TRAIN_CONFIG['n_trials'],
    n_folds=TRAIN_CONFIG['n_folds'],
    n_test_folds=TRAIN_CONFIG['n_test_folds'],
    purged_size=TRAIN_CONFIG['purged_size'],
    embargo_size=TRAIN_CONFIG['embargo_size'],
    seed=TRAIN_CONFIG['seed'],
    device=TRAIN_CONFIG['device'],
    min_trials=TRAIN_CONFIG['min_trials'],
)

print(f"Pipeline initialized with:")
print(f"  - Test size: {TRAIN_CONFIG['test_size']}")
print(f"  - Optuna trials per model: {TRAIN_CONFIG['n_trials']}")
print(f"  - CV folds: {TRAIN_CONFIG['n_folds']}")
print(f"  - Purge/Embargo: {TRAIN_CONFIG['purged_size']}/{TRAIN_CONFIG['embargo_size']} days")
print(f"  - Device: {TRAIN_CONFIG['device']}\n")

# Train the models (LightGBM, XGBoost, CatBoost)
print("Training models with Optuna hyperparameter optimization...")
pipeline.fit(df, save_path=TRAIN_CONFIG['save_model_path'])

# Display comprehensive results
print("\n" + "="*80)
print("FINAL TRAINING RESULTS")
print("="*80)
print(f"✓ Best Model: {pipeline.model_selector_.best_model_name_}")
print(f"✓ Best NDCG@100 Score: {pipeline.model_selector_.best_score_:.6f}")
print(f"✓ Training Sharpe Ratio: {pipeline.train_score_:.6f}")
print(f"✓ Validation Sharpe Ratio: {pipeline.val_score_:.6f}")

# Display detailed results for all models (using correct dictionary keys)
print("\n" + "-"*80)
print("DETAILED MODEL COMPARISON")
print("-"*80)
for model_name, results in pipeline.model_selector_.model_results_.items():
    print(f"\n{model_name}:")
    print(f"  ├─ NDCG@100:          {results['best_ndcg']:.6f}")
    print(f"  ├─ Best Sharpe:       {results['best_sharpe']:.6f}")
    print(f"  ├─ Trials Completed:  {results['n_trials_completed']}")
    print(f"  └─ Best Params:       {results['best_params']}")

print("\n" + "="*80)
print(f"✓ Model saved to: {TRAIN_CONFIG['save_model_path']}")
print("="*80)

[I 2026-01-12 19:30:21,604] A new study created in memory with name: no-name-5356ca76-8445-4e5a-86c2-93b60d1ede19


STARTING MODEL TRAINING WITH UPDATED PIPELINE
Training data shape: (249159, 13)
Features: ['Date', 'SecuritiesCode', 'Target', 'vwap_rate', 'side', 'vol5', 'vol10', 'vol15', 'vol21', 'autocorr-10', 'autocorr-15', 'autocorr-21', 'Rank']

Pipeline initialized with:
  - Test size: 0.2
  - Optuna trials per model: 5
  - CV folds: 5
  - Purge/Embargo: 2/21 days
  - Device: cpu

Training models with Optuna hyperparameter optimization...
TRAINING PIPELINE START

[1/4] Splitting data...
Train samples: 211,375 | Val samples: 37,784
Train dates: 2017-01-12 00:00:00 to 2020-12-07 00:00:00
Val dates: 2020-12-08 00:00:00 to 2021-11-29 00:00:00

[2/4] Training models...

Training LightGBM with CPCV + Optuna
Cumulative trials so far: 0


  0%|          | 0/5 [00:00<?, ?it/s]

Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[35]	valid_0's ndcg@10: 0.986989	valid_0's ndcg@50: 0.98739	valid_0's ndcg@100: 0.991231
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[35]	valid_0's ndcg@10: 0.970454	valid_0's ndcg@50: 0.965747	valid_0's ndcg@100: 0.969126
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[44]	valid_0's ndcg@10: 0.984417	valid_0's ndcg@50: 0.986311	valid_0's ndcg@100: 0.990473
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[42]	valid_0's ndcg@10: 0.949	valid_0's ndcg@50: 0.937659	valid_0's ndcg@100: 0.937966
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[22]	valid_0's ndcg@10: 0.962387	valid_0's ndcg@50: 0.955303	valid_0's ndcg@100: 0.96107
Training until validation scores don't improve for 30 rounds
Early stopping

2026-01-12 19:30:47.023 | DEBUG    | jpx_ranker._src.model_selection:__call__:86 - MinTRL: Collecting trials (1/5 needed before stopping check)


...

[I 2026-01-12 19:48:31,963] Trial 2 finished with values: [0.9907342067045735, 0.14319614886860216] and parameters: {'learning_rate': 0.06272924049005918, 'iterations': 200, 'depth': 6, 'l2_leaf_reg': 5.72280788469014, 'border_count': 128, 'bagging_temperature': 2.9122914019804194, 'random_strength': 0.0032112643094417484}.


2026-01-12 19:54:17.573 | DEBUG    | jpx_ranker._src.model_selection:__call__:145 - MinTRL: Significance reached but model needs more exploration (4/5)...


[I 2026-01-12 19:54:17,567] Trial 3 finished with values: [0.9907647539887483, 0.1461740706757229] and parameters: {'learning_rate': 0.05045321958909213, 'iterations': 300, 'depth': 6, 'l2_leaf_reg': 5.104629857953324, 'border_count': 207, 'bagging_temperature': 1.9967378215835974, 'random_strength': 0.00042472707398058225}.


2026-01-12 19:58:18.953 | DEBUG    | jpx_ranker._src.model_selection:__call__:140 - MinTRL Status (K_global=15, K_model=5): Best SR=0.9939, PSR=1.000, T=380/3 needed
2026-01-12 19:58:18.954 | INFO     | jpx_ranker._src.model_selection:__call__:149 - 
✓ Early stopping: no-name-dfe825f3-040c-452f-8d08-d319066b8fee reached MinTRL significance. (T=380, PSR=1.000, K_global=15, K_model=5)


[I 2026-01-12 19:58:18,946] Trial 4 finished with values: [0.9905081873358714, 0.13931994925325486] and parameters: {'learning_rate': 0.18180022496999232, 'iterations': 100, 'depth': 8, 'l2_leaf_reg': 2.5347171131856236, 'border_count': 46, 'bagging_temperature': 9.488855372533333, 'random_strength': 4.905556676028774}.

Best NDCG: 0.9905
Predicted Sharpe: 0.9526
Trials completed: 5/5
Best No. of Trials: 3

STATISTICAL VALIDATION REPORT
Best Model: LightGBM
Best NDCG: 0.9563
Best Sharpe Difference (minimized): 0.9563
Predicted Sharpe Ratio (Daily): 0.9822
Annualized Sharpe Ratio: 15.5916

⚠️  LEAKAGE WARNING: Annualized SR (15.59) is institutionally impossible!
    This indicates the competition's target data contains inherent lookahead bias.
    Even with CPCV purging/embargoing, the provided 'Target' column uses future data.
    Results are valid for competition ranking but NOT for live trading.

Raw Sharpe Ratio: 0.9822
Expected Max SR (haircut): 0.0326
Deflated Sharpe Ratio (DSR): 

## Model Inference

In [12]:
from jpx_ranker.kaggle_submission import run_kaggle_submission

In [13]:
run_kaggle_submission(
    model_path='kaggle/working/best_ranker_model.pkl',
    fast_period=5,
    slow_period=10,
    enable_submission=False  # Test mode
)


KAGGLE SUBMISSION: DISABLED (Local Testing Mode)

To enable Kaggle submission:
  1. Set enable_submission=True
  2. Ensure the model is saved at: kaggle/working/best_ranker_model.pkl
  3. Run this in the Kaggle notebook environment

For local testing, use the InferencePipeline class directly:
  >>> from jpx_ranker._src.infer import InferencePipeline
  >>> inference_pipeline = InferencePipeline(model=trained_model)
  >>> predictions = inference_pipeline.predict(test_data)


In [14]:
from jpx_ranker._src.infer import InferencePipeline
from jpx_ranker._src.utils import calc_spread_return_sharpe, calc_spread_return_sharpe_scorer

# The pipeline automatically extracts the best model and the matching data processor
inference_pipeline = InferencePipeline(model=pipeline) 
predictions = inference_pipeline.predict(df_test)
predictions

Compute Securities Features: 100%|#########9| 1995/2000 [04:06<00:00, 10.74it/s]

,Date,SecuritiesCode,Target,Rank,Score,PredictedRank,Close,VWAP,vwap_rate,side,vol5,vol10,vol15,vol21,autocorr-10,autocorr-15,autocorr-21
0,2021-12-08,7826,0.119247,0,1.606963,1,9700.0,9588.963848,0.055264,1.0,0.072416,0.072416,0.072416,0.072416,-1.000000,-1.000000,-1.000000
1,2021-12-08,4772,0.088889,1,1.995340,0,180.0,179.983900,0.086838,1.0,0.052987,0.052987,0.052987,0.052987,1.000000,1.000000,1.000000
2,2021-12-08,6047,0.086675,2,1.438614,4,754.0,732.587817,0.066521,1.0,0.030920,0.030920,0.030920,0.030920,-1.000000,-1.000000,-1.000000
3,2021-12-08,4550,0.080235,3,1.580517,2,1014.0,996.245314,0.065648,1.0,0.040121,0.040121,0.040121,0.040121,-1.000000,-1.000000,-1.000000
4,2021-12-08,9086,0.076923,4,1.458721,3,5260.0,5196.699087,0.046568,1.0,0.047622,0.047622,0.047622,0.047622,-1.000000,-1.000000,-1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260173,2022-06-22,4541,-0.038356,1982,-0.615494,1592,358.0,429.399518,-0.000812,1.0,0.026276,0.021791,0.026061,0.045182,-0.157956,0.073655,-0.163941
260174,2022-06-22,7013,-0.052980,1983,0.410328,10,3810.0,2982.456201,0.002882,1.0,0.040687,0.027344,0.030610,0.031913,-0.154657,0.018904,-0.054306
260175,2022-06-22,4883,-0.060491,1984,0.301621,19,449.0,449.098739,0.002556,1.0,0.103726,0.089553,0.073995,0.077269,-0.247704,-0.266207,-0.300761
260176,2022-06-22,7211,-0.086066,1985,0.188644,45,482.0,343.070212,0.005535,-1.0,0.057006,0.045054,0.039094,0.034313,-0.233474,-0.085598,-0.118358


In [15]:
eval_df = predictions[['Date', 'SecuritiesCode', 'Target', 'Rank', 'Score', 'PredictedRank']].sort_values(by=['Date', 'PredictedRank'])
eval_df = eval_df.drop(columns=['Rank'])
eval_df = eval_df.rename(columns={'PredictedRank': 'Rank'})
eval_df

,Date,SecuritiesCode,Target,Score,Rank
1,2021-12-08,4772,0.088889,1.995340,0
0,2021-12-08,7826,0.119247,1.606963,1
3,2021-12-08,4550,0.080235,1.580517,2
4,2021-12-08,9086,0.076923,1.458721,3
2,2021-12-08,6047,0.086675,1.438614,4
...,...,...,...,...,...
258426,2022-06-22,4495,0.038404,-0.876307,1982
260115,2022-06-22,4229,-0.018592,-0.878978,1983
258455,2022-06-22,4828,0.035764,-0.897057,1984
258194,2022-06-22,4167,0.149031,-0.934947,1985


In [16]:
sharpe = calc_spread_return_sharpe(eval_df)
print(f"Test Sharpe Ratio: {sharpe:.4f}")

Test Sharpe Ratio: 1.4424
